# TFT Model Improvements Experiment

Testing the following changes to improve direction accuracy:

1. **Target = returns** instead of raw close prices
2. **Direction-aware composite loss** (QuantileLoss + DirectionLoss)
3. **Feature correlation filtering** (0.95 threshold)
4. **Walk-forward cross-validation** (rolling window)
5. **Larger model capacity** (hidden_size=64, attention_head_size=4)
6. **Checkpoint ensemble** (top-3 averaging)

Each improvement is tested incrementally against the baseline to measure impact.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

from pytorch_forecasting import TimeSeriesDataSet, GroupNormalizer, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint

from pipeline.features import calculate_all_features

# Reproducibility
pl.seed_everything(42)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Seed set to 42


PyTorch: 2.5.1+cu121
Device: cuda


## 1. Load Data (same as existing notebook)

In [2]:
import requests
import time

# Load API key
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ.get('POLYGON_API_KEY')
assert api_key, "Set POLYGON_API_KEY in .env"

def fetch_polygon_1min_data(symbol, start_date, end_date, api_key):
    all_data = []
    url = f"https://api.polygon.io/v2/aggs/ticker/{symbol}/range/1/minute/{start_date}/{end_date}"
    params = {"adjusted": "true", "sort": "asc", "limit": 50000, "apiKey": api_key}
    while url:
        resp = requests.get(url, params=params, timeout=60)
        data = resp.json()
        if data.get("status") not in ("OK", "DELAYED") or "results" not in data:
            break
        all_data.extend(data["results"])
        print(f"  Fetched {len(all_data)} bars...", end="\r")
        next_url = data.get("next_url")
        if next_url:
            url = next_url
            params = {"apiKey": api_key}
            time.sleep(0.5)
        else:
            url = None
    df = pd.DataFrame(all_data)
    df["timestamp"] = pd.to_datetime(df["t"], unit="ms", utc=True)
    df["timestamp"] = df["timestamp"].dt.tz_convert("America/New_York").dt.tz_localize(None)
    df = df.rename(columns={"o": "open", "h": "high", "l": "low", "c": "close", "v": "volume"})
    df = df[["timestamp", "open", "high", "low", "close", "volume"]]
    df = df.sort_values("timestamp").reset_index(drop=True)
    print(f"\nLoaded {len(df)} bars: {df['timestamp'].min()} to {df['timestamp'].max()}")
    return df

# Fetch 420 days of SPY data
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=420)).strftime("%Y-%m-%d")
df_raw = fetch_polygon_1min_data("SPY", start_date, end_date, api_key)
df_raw.head()

  Fetched 250000 bars...
Loaded 250000 bars: 2025-03-05 04:00:00 to 2026-04-24 06:51:00


,timestamp,open,high,low,close,volume
0,2025-03-05 04:00:00,580.70,580.70,579.94,579.94,10440.0
1,2025-03-05 04:01:00,580.18,580.28,580.18,580.24,5903.0
2,2025-03-05 04:02:00,580.17,580.17,580.10,580.12,2845.0
3,2025-03-05 04:03:00,580.22,580.22,580.14,580.14,584.0
4,2025-03-05 04:04:00,580.28,580.34,580.19,580.19,1313.0


## 2. Feature Engineering + Correlation Filtering

In [6]:
# Calculate VWAP (as in main.py)
df_raw['vwap'] = (df_raw['volume'] * (df_raw['high'] + df_raw['low'] + df_raw['close']) / 3).cumsum() / df_raw['volume'].cumsum()

# Feature engineering (76 features)
df_features = calculate_all_features(df_raw)
print(f"Features calculated: {df_features.shape}")

# --- IMPROVEMENT #3: Correlation Filtering ---
def remove_highly_correlated_features(df, threshold=0.95):
    """Remove features with correlation > threshold (matching notebook logic)."""
    exclude_cols = ['timestamp', 'time_idx', 'group', 'target_close_60m', 'target_return_60m',
                    'open', 'high', 'low', 'close', 'volume', 'vwap']
    feature_cols = [c for c in df.columns if c not in exclude_cols and df[c].dtype in ['float64', 'float32', 'int64']]
    
    df_clean = df[feature_cols].dropna()
    corr_matrix = df_clean.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    
    print(f"Correlation threshold: {threshold}")
    print(f"Features before filtering: {len(feature_cols)}")
    print(f"Features to drop: {len(to_drop)}")
    print(f"Features remaining: {len(feature_cols) - len(to_drop)}")
    
    df_filtered = df.drop(columns=to_drop)
    return df_filtered, to_drop, corr_matrix

df_filtered, dropped_cols, corr_matrix = remove_highly_correlated_features(df_features, threshold=0.95)
print(f"\nDropped features: {dropped_cols}")
print(f"\nFiltered shape: {df_filtered.shape}")

Features calculated: (250000, 84)
Correlation threshold: 0.95
Features before filtering: 72
Features to drop: 4
Features remaining: 68

Dropped features: ['sma_5_slope', 'sma_30_slope', 'roc_10', 'roc_20']

Filtered shape: (250000, 80)


In [ ]:
corr_matrix[corr_matrix > 0.7]

,returns_1m,returns_5m,returns_15m,returns_30m,returns_60m,high_low_ratio,close_open_ratio,high_close_ratio,low_close_ratio,upper_shadow,...,hour_cos,minute_sin,minute_cos,day_sin,day_cos,volume_lag_1,volume_lag_5,volume_lag_15,volume_lag_30,volume_lag_60
returns_1m,1.0,NaN,NaN,NaN,NaN,NaN,0.876516,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
returns_5m,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
returns_15m,NaN,NaN,1.000000,0.692684,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
returns_30m,NaN,NaN,0.692684,1.000000,0.699721,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
returns_60m,NaN,NaN,NaN,0.699721,1.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
volume_lag_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
volume_lag_5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
volume_lag_15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
volume_lag_30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN


## 3. Prepare Dataset

Two variants:
- **Baseline**: target = `close` (current approach)
- **Improved**: target = `close_return` (percent change from previous bar)

In [10]:
def prepare_tft_data(df, use_returns_target=False):
    """Prepare TFT-ready DataFrame with optional returns target."""
    df = df.copy()
    
    # Drop NaNs from feature calculation
    target_cols = ['target_close_60m', 'target_return_60m']
    non_target = [c for c in df.columns if c not in target_cols]
    df = df.dropna(subset=non_target)
    df = df.dropna(subset=target_cols)
    df = df.reset_index(drop=True)
    
    # --- IMPROVEMENT #1: Returns target ---
    if use_returns_target:
        # Create a returns-based target: percent change from each bar's close
        # This is what the model will predict step-by-step in the forecast horizon
        df['close_return'] = df['close'].pct_change() * 100  # basis points scale
        df = df.dropna(subset=['close_return'])
        df = df.reset_index(drop=True)
    
    # Required TFT columns
    df['time_idx'] = range(len(df))
    df['group'] = 'SPY'
    
    # Clean infinities
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df[numeric_cols] = df[numeric_cols].ffill().bfill()
    
    target = 'close_return' if use_returns_target else 'close'
    print(f"Prepared dataset: {len(df)} rows, target='{target}'")
    return df, target


def build_feature_lists(df, target):
    """Build time-varying known/unknown feature lists."""
    time_varying_known_reals = ['hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'day_sin', 'day_cos']
    time_varying_known_reals = [c for c in time_varying_known_reals if c in df.columns]
    
    exclude = [
        'timestamp', 'time_idx', 'group',
        'target_close_60m', 'target_return_60m',
        'hour', 'minute', 'day_of_week',
        'is_morning', 'is_afternoon',
        'close_return',  # don't include returns target in unknown reals if it's the target
    ] + time_varying_known_reals
    
    time_varying_unknown_reals = [c for c in df.columns if c not in exclude]
    
    # Ensure target is first
    if target in time_varying_unknown_reals:
        time_varying_unknown_reals.remove(target)
    time_varying_unknown_reals = [target] + time_varying_unknown_reals
    
    print(f"Known reals: {len(time_varying_known_reals)}, Unknown reals: {len(time_varying_unknown_reals)}")
    return time_varying_known_reals, time_varying_unknown_reals


# Prepare both variants
df_baseline, target_baseline = prepare_tft_data(df_filtered, use_returns_target=False)
df_returns, target_returns = prepare_tft_data(df_filtered, use_returns_target=True)

known_reals_b, unknown_reals_b = build_feature_lists(df_baseline, target_baseline)
known_reals_r, unknown_reals_r = build_feature_lists(df_returns, target_returns)

Prepared dataset: 249820 rows, target='close'
Prepared dataset: 249819 rows, target='close_return'
Known reals: 6, Unknown reals: 66
Known reals: 6, Unknown reals: 67


## 4. Direction-Aware Composite Loss

Combines QuantileLoss with a penalty for getting the direction wrong.
The direction component penalizes predictions where `sign(pred_change) != sign(actual_change)`.

In [11]:
from pytorch_forecasting.metrics import MultiHorizonMetric


class DirectionAwareQuantileLoss(MultiHorizonMetric):
    """
    Composite loss: QuantileLoss + direction penalty.
    
    For returns target:
      direction_loss = penalty when sign(predicted_return) != sign(actual_return)
    
    For price target:
      direction_loss = penalty when sign(pred[t] - pred[t-1]) != sign(actual[t] - actual[t-1])
    
    Args:
        quantile_weight: Weight for the QuantileLoss component (default 0.7)
        direction_weight: Weight for the direction penalty (default 0.3)
        quantiles: List of quantiles for QuantileLoss
    """
    def __init__(
        self,
        quantile_weight: float = 0.7,
        direction_weight: float = 0.3,
        quantiles: list = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98],
        **kwargs,
    ):
        super().__init__(quantiles=quantiles, **kwargs)
        self.quantile_weight = quantile_weight
        self.direction_weight = direction_weight
        self.quantiles = quantiles

    def loss(self, y_pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        Compute composite loss.
        y_pred: (batch, horizon, n_quantiles)
        target: (batch, horizon)
        """
        # --- Quantile loss component ---
        losses = []
        for i, q in enumerate(self.quantiles):
            errors = target - y_pred[..., i]
            q_loss = torch.max((q - 1) * errors, q * errors)
            losses.append(q_loss.unsqueeze(-1))
        quantile_loss = torch.cat(losses, dim=-1).mean(dim=-1)  # (batch, horizon)
        
        # --- Direction penalty component ---
        # Use median quantile for direction assessment
        median_idx = len(self.quantiles) // 2
        pred_median = y_pred[..., median_idx]  # (batch, horizon)
        
        # Direction: compare consecutive steps in the prediction horizon
        # pred_change[t] = pred[t] - pred[t-1], actual_change[t] = actual[t] - actual[t-1]
        pred_change = pred_median[:, 1:] - pred_median[:, :-1]
        actual_change = target[:, 1:] - target[:, :-1]
        
        # Soft direction penalty: high when signs disagree
        # Using tanh product: negative when directions disagree
        direction_agreement = torch.tanh(pred_change * 10) * torch.tanh(actual_change * 10)
        direction_penalty = torch.clamp(1.0 - direction_agreement, min=0.0) / 2.0  # 0 when agree, 1 when disagree
        
        # Pad to match horizon length (first step has no direction)
        pad = torch.zeros_like(direction_penalty[:, :1])
        direction_penalty = torch.cat([pad, direction_penalty], dim=1)  # (batch, horizon)
        
        # Combine
        combined = self.quantile_weight * quantile_loss + self.direction_weight * direction_penalty
        return combined


# Quick test
loss_fn = DirectionAwareQuantileLoss(quantile_weight=0.7, direction_weight=0.3)
print(f"DirectionAwareQuantileLoss created with quantiles={loss_fn.quantiles}")
print(f"Weights: quantile={loss_fn.quantile_weight}, direction={loss_fn.direction_weight}")

DirectionAwareQuantileLoss created with quantiles=[0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
Weights: quantile=0.7, direction=0.3


## 5. Walk-Forward Cross-Validation Framework

Instead of a single 80/20 split, we use rolling windows:
- Train on N months, validate on the next month
- Slide forward and repeat
- Report average direction accuracy across all folds

In [12]:
def create_walk_forward_splits(df, n_folds=5, val_fraction=0.1):
    """
    Create walk-forward train/val splits.
    
    Each fold uses an expanding training window with a fixed validation window.
    
    Returns list of (train_cutoff_idx, val_start_idx, val_end_idx) tuples.
    """
    total_rows = len(df)
    val_size = int(total_rows * val_fraction)
    
    # Reserve the last n_folds * val_size rows for validation windows
    min_train_size = int(total_rows * 0.5)  # At least 50% for first training window
    
    splits = []
    for fold in range(n_folds):
        val_end = total_rows - (n_folds - fold - 1) * val_size
        val_start = val_end - val_size
        train_end = val_start  # training goes up to val_start
        
        if train_end < min_train_size:
            continue
        
        splits.append({
            'fold': fold,
            'train_end_idx': train_end,
            'val_start_idx': val_start,
            'val_end_idx': val_end,
            'train_rows': train_end,
            'val_rows': val_end - val_start,
        })
    
    print(f"Walk-forward splits ({len(splits)} folds):")
    for s in splits:
        print(f"  Fold {s['fold']}: train=[0:{s['train_end_idx']}] ({s['train_rows']} rows), "
              f"val=[{s['val_start_idx']}:{s['val_end_idx']}] ({s['val_rows']} rows)")
    return splits


# Preview splits
splits = create_walk_forward_splits(df_returns, n_folds=5, val_fraction=0.08)
print(f"\nTotal rows: {len(df_returns)}")

Walk-forward splits (5 folds):
  Fold 0: train=[0:149894] (149894 rows), val=[149894:169879] (19985 rows)
  Fold 1: train=[0:169879] (169879 rows), val=[169879:189864] (19985 rows)
  Fold 2: train=[0:189864] (189864 rows), val=[189864:209849] (19985 rows)
  Fold 3: train=[0:209849] (209849 rows), val=[209849:229834] (19985 rows)
  Fold 4: train=[0:229834] (229834 rows), val=[229834:249819] (19985 rows)

Total rows: 249819


## 6. Training & Evaluation Harness

A unified function that trains a TFT model with a given configuration and evaluates
direction accuracy at 15/30/45/60 minute horizons.

In [13]:
def evaluate_direction_accuracy(model, val_dataloader, target_name, df_val=None):
    """
    Evaluate direction accuracy at key horizons.
    
    For returns target: direction = sign of predicted return
    For price target: direction = sign of (predicted_price - base_price)
    
    Returns dict with accuracy at each horizon.
    """
    predictions = model.predict(val_dataloader, return_x=True)
    
    # Extract median prediction
    if predictions.output.dim() == 3:
        pred = predictions.output[:, :, predictions.output.shape[2] // 2]  # median
    else:
        pred = predictions.output
    
    actual = predictions.x["decoder_target"]
    if actual.dim() == 3:
        actual = actual[:, :, 0]
    
    pred_np = pred.detach().cpu().numpy()
    actual_np = actual.detach().cpu().numpy()
    
    results = {}
    horizons = {15: 14, 30: 29, 45: 44, 60: 59}  # minute -> index
    
    for horizon_min, idx in horizons.items():
        if idx >= pred_np.shape[1]:
            continue
        
        if target_name == 'close_return':
            # For returns: sum the predicted returns over the horizon to get cumulative return
            pred_cumulative = pred_np[:, :idx+1].sum(axis=1)
            actual_cumulative = actual_np[:, :idx+1].sum(axis=1)
            pred_direction = np.sign(pred_cumulative)
            actual_direction = np.sign(actual_cumulative)
        else:
            # For price: direction relative to the first predicted value (base)
            # The encoder target's last value is the base price
            encoder_target = predictions.x["encoder_target"]
            if encoder_target.dim() == 3:
                encoder_target = encoder_target[:, :, 0]
            base = encoder_target[:, -1].detach().cpu().numpy()
            pred_direction = np.sign(pred_np[:, idx] - base)
            actual_direction = np.sign(actual_np[:, idx] - base)
        
        # Direction accuracy (exclude flat cases)
        mask = (pred_direction != 0) & (actual_direction != 0)
        if mask.sum() > 0:
            correct = (pred_direction[mask] == actual_direction[mask]).mean()
        else:
            correct = 0.5
        
        results[f'dir_acc_{horizon_min}m'] = float(correct)
    
    # Also compute MAE and val_loss for reference
    pred_flat = pred_np.flatten()
    actual_flat = actual_np.flatten()
    results['mae'] = float(np.mean(np.abs(pred_flat - actual_flat)))
    results['rmse'] = float(np.sqrt(np.mean((pred_flat - actual_flat) ** 2)))
    
    return results


print("Evaluation harness ready.")

Evaluation harness ready.


In [ ]:
def train_and_evaluate(
    df,
    target,
    known_reals,
    unknown_reals,
    train_cutoff,
    loss_fn=None,
    hidden_size=32,
    attention_head_size=2,
    hidden_continuous_size=16,
    max_epochs=25,
    batch_size=64,
    learning_rate=0.001,
    dropout=0.1,
    patience=10,
    save_top_k=1,
    experiment_name="experiment",
):
    """
    Train a TFT model and return evaluation metrics + trained model.
    """
    max_encoder_length = 60
    max_prediction_length = 60
    
    if loss_fn is None:
        loss_fn = QuantileLoss()
    
    # Use appropriate normalizer based on target type:
    # - softplus for price targets (always positive)
    # - no transformation for returns (can be negative)
    if target == 'close_return':
        target_normalizer = GroupNormalizer(groups=["group"])
    else:
        target_normalizer = GroupNormalizer(groups=["group"], transformation="softplus")
    
    # Build datasets
    training = TimeSeriesDataSet(
        df[lambda x: x.time_idx <= train_cutoff],
        time_idx="time_idx",
        target=target,
        group_ids=["group"],
        min_encoder_length=max_encoder_length // 2,
        max_encoder_length=max_encoder_length,
        max_prediction_length=max_prediction_length,
        static_categoricals=["group"],
        time_varying_known_categoricals=[],
        time_varying_known_reals=known_reals,
        time_varying_unknown_categoricals=[],
        time_varying_unknown_reals=unknown_reals,
        target_normalizer=target_normalizer,
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )
    
    validation = TimeSeriesDataSet.from_dataset(
        training,
        df[lambda x: x.time_idx > train_cutoff],
        predict=False,
    )
    
    train_dl = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_dl = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=0)
    
    # Build model
    tft = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=learning_rate,
        hidden_size=hidden_size,
        attention_head_size=attention_head_size,
        dropout=dropout,
        hidden_continuous_size=hidden_continuous_size,
        output_size=7,  # 7 quantiles
        loss=loss_fn,
        reduce_on_plateau_patience=4,
    )
    print(f"[{experiment_name}] Model params: {tft.size()/1e3:.1f}k")
    
    # Callbacks (no LearningRateMonitor since logger=False)
    ckpt_dir = f"/tmp/checkpoints/{experiment_name}"
    os.makedirs(ckpt_dir, exist_ok=True)
    
    callbacks = [
        EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=patience, verbose=False, mode="min"),
        ModelCheckpoint(
            dirpath=ckpt_dir,
            monitor="val_loss",
            filename="tft-{epoch:02d}-{val_loss:.4f}",
            save_top_k=save_top_k,
            mode="min",
        ),
    ]
    
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        callbacks=callbacks,
        enable_model_summary=False,
        enable_progress_bar=True,
        accelerator="auto",
        gradient_clip_val=0.1,
        logger=False,
    )
    
    trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)
    
    # Load best checkpoint
    best_path = callbacks[1].best_model_path
    best_val_loss = float(callbacks[1].best_model_score)
    best_model = TemporalFusionTransformer.load_from_checkpoint(best_path)
    best_model.eval()
    
    # Evaluate
    metrics = evaluate_direction_accuracy(best_model, val_dl, target)
    metrics['val_loss'] = best_val_loss
    metrics['epochs'] = trainer.current_epoch + 1
    metrics['experiment'] = experiment_name
    
    print(f"[{experiment_name}] val_loss={best_val_loss:.4f}, "
          f"dir_15m={metrics.get('dir_acc_15m', 0):.1%}, "
          f"dir_30m={metrics.get('dir_acc_30m', 0):.1%}, "
          f"dir_60m={metrics.get('dir_acc_60m', 0):.1%}")
    
    return best_model, metrics, training, callbacks[1]


print("Training harness ready.")

Training harness ready.


## 7. Run Experiments

### Experiment A: Baseline (current config, single split)

In [15]:
# Baseline: close target, QuantileLoss, hidden=32, no correlation filtering
# (using filtered data but same architecture as current production)
train_cutoff_b = int(len(df_baseline) * 0.8)

print("=" * 60)
print("EXPERIMENT A: Baseline (close target, QuantileLoss, hidden=32)")
print("=" * 60)

model_a, metrics_a, _, _ = train_and_evaluate(
    df=df_baseline,
    target=target_baseline,
    known_reals=known_reals_b,
    unknown_reals=unknown_reals_b,
    train_cutoff=train_cutoff_b,
    loss_fn=QuantileLoss(),
    hidden_size=32,
    attention_head_size=2,
    max_epochs=25,
    experiment_name="A_baseline",
)
print(f"\nBaseline results: {metrics_a}")

EXPERIMENT A: Baseline (close target, QuantileLoss, hidden=32)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 3060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


[A_baseline] Model params: 251.0k


MisconfigurationException: Cannot use `LearningRateMonitor` callback with `Trainer` that has no logger.

### Experiment B: Returns Target (Improvement #1)

In [16]:
# Returns target with standard QuantileLoss
train_cutoff_r = int(len(df_returns) * 0.8)

print("=" * 60)
print("EXPERIMENT B: Returns Target (close_return, QuantileLoss, hidden=32)")
print("=" * 60)

model_b, metrics_b, _, _ = train_and_evaluate(
    df=df_returns,
    target=target_returns,
    known_reals=known_reals_r,
    unknown_reals=unknown_reals_r,
    train_cutoff=train_cutoff_r,
    loss_fn=QuantileLoss(),
    hidden_size=32,
    attention_head_size=2,
    max_epochs=25,
    experiment_name="B_returns_target",
)
print(f"\nReturns target results: {metrics_b}")

EXPERIMENT B: Returns Target (close_return, QuantileLoss, hidden=32)


ValueError: 94572 (47.32%) of close_return values were found to be NA or infinite (even after encoding). NA values are not allowed `allow_missing_timesteps` refers to missing rows, not to missing values. Possible strategies to fix the issue are (a) dropping the variable close_return, (b) using `NaNLabelEncoder(add_nan=True)` for categorical variables, (c) filling missing values and/or (d) optionally adding a variable indicating filled values

### Experiment C: Returns + Direction-Aware Loss (Improvements #1 + #2)

In [ ]:
print("=" * 60)
print("EXPERIMENT C: Returns + DirectionAwareQuantileLoss")
print("=" * 60)

model_c, metrics_c, _, _ = train_and_evaluate(
    df=df_returns,
    target=target_returns,
    known_reals=known_reals_r,
    unknown_reals=unknown_reals_r,
    train_cutoff=train_cutoff_r,
    loss_fn=DirectionAwareQuantileLoss(quantile_weight=0.7, direction_weight=0.3),
    hidden_size=32,
    attention_head_size=2,
    max_epochs=25,
    experiment_name="C_direction_loss",
)
print(f"\nDirection-aware loss results: {metrics_c}")

### Experiment D: Returns + Direction Loss + Larger Model (Improvements #1 + #2 + #5)

In [ ]:
print("=" * 60)
print("EXPERIMENT D: Returns + DirectionLoss + Larger Model (64/4)")
print("=" * 60)

model_d, metrics_d, _, ckpt_cb_d = train_and_evaluate(
    df=df_returns,
    target=target_returns,
    known_reals=known_reals_r,
    unknown_reals=unknown_reals_r,
    train_cutoff=train_cutoff_r,
    loss_fn=DirectionAwareQuantileLoss(quantile_weight=0.7, direction_weight=0.3),
    hidden_size=64,
    attention_head_size=4,
    hidden_continuous_size=32,
    max_epochs=25,
    save_top_k=3,  # Save top 3 for ensemble (Improvement #6)
    experiment_name="D_larger_model",
)
print(f"\nLarger model results: {metrics_d}")

### Experiment E: Checkpoint Ensemble (Improvement #6)

Average predictions from top-3 checkpoints of the best configuration.

In [ ]:
def ensemble_evaluate(checkpoint_callback, val_dataloader, target_name):
    """
    Load top-k checkpoints, average their median predictions, evaluate direction accuracy.
    """
    # Get all saved checkpoint paths
    ckpt_paths = checkpoint_callback.best_k_models
    if not ckpt_paths:
        print("No checkpoints found for ensemble.")
        return None
    
    print(f"Ensemble over {len(ckpt_paths)} checkpoints:")
    for path, score in ckpt_paths.items():
        print(f"  {os.path.basename(path)}: val_loss={score:.4f}")
    
    all_preds = []
    first_predictions = None
    
    for path in ckpt_paths.keys():
        model = TemporalFusionTransformer.load_from_checkpoint(path)
        model.eval()
        predictions = model.predict(val_dataloader, return_x=True)
        
        if predictions.output.dim() == 3:
            pred = predictions.output[:, :, predictions.output.shape[2] // 2]
        else:
            pred = predictions.output
        
        all_preds.append(pred.detach().cpu().numpy())
        if first_predictions is None:
            first_predictions = predictions
    
    # Average predictions
    ensemble_pred = np.mean(all_preds, axis=0)
    
    # Get actuals
    actual = first_predictions.x["decoder_target"]
    if actual.dim() == 3:
        actual = actual[:, :, 0]
    actual_np = actual.detach().cpu().numpy()
    
    # Evaluate direction accuracy
    results = {}
    horizons = {15: 14, 30: 29, 45: 44, 60: 59}
    
    for horizon_min, idx in horizons.items():
        if idx >= ensemble_pred.shape[1]:
            continue
        
        if target_name == 'close_return':
            pred_cumulative = ensemble_pred[:, :idx+1].sum(axis=1)
            actual_cumulative = actual_np[:, :idx+1].sum(axis=1)
            pred_dir = np.sign(pred_cumulative)
            actual_dir = np.sign(actual_cumulative)
        else:
            encoder_target = first_predictions.x["encoder_target"]
            if encoder_target.dim() == 3:
                encoder_target = encoder_target[:, :, 0]
            base = encoder_target[:, -1].detach().cpu().numpy()
            pred_dir = np.sign(ensemble_pred[:, idx] - base)
            actual_dir = np.sign(actual_np[:, idx] - base)
        
        mask = (pred_dir != 0) & (actual_dir != 0)
        if mask.sum() > 0:
            correct = (pred_dir[mask] == actual_dir[mask]).mean()
        else:
            correct = 0.5
        results[f'dir_acc_{horizon_min}m'] = float(correct)
    
    results['mae'] = float(np.mean(np.abs(ensemble_pred.flatten() - actual_np.flatten())))
    results['experiment'] = 'E_ensemble'
    return results


# Build validation dataloader for experiment D's data
val_ds_d = TimeSeriesDataSet.from_parameters(
    model_d.dataset_parameters,
    df_returns[lambda x: x.time_idx > train_cutoff_r],
    predict=False,
)
val_dl_d = val_ds_d.to_dataloader(train=False, batch_size=64, num_workers=0)

print("=" * 60)
print("EXPERIMENT E: Checkpoint Ensemble (top-3 from Exp D)")
print("=" * 60)

metrics_e = ensemble_evaluate(ckpt_cb_d, val_dl_d, target_returns)
if metrics_e:
    print(f"\nEnsemble results: {metrics_e}")

## 8. Walk-Forward Validation (Improvement #4)

Run the best configuration across multiple time periods to validate robustness.

In [ ]:
# Walk-forward validation with the best config so far
# (This takes a while - trains one model per fold)

splits = create_walk_forward_splits(df_returns, n_folds=3, val_fraction=0.10)

wf_results = []
for split in splits:
    fold = split['fold']
    print(f"\n{'=' * 60}")
    print(f"WALK-FORWARD FOLD {fold}")
    print(f"{'=' * 60}")
    
    # Use expanding window: train on [0, train_end], validate on [val_start, val_end]
    df_fold = df_returns[df_returns['time_idx'] < split['val_end_idx']].copy()
    df_fold['time_idx'] = range(len(df_fold))
    
    fold_cutoff = split['train_end_idx']
    
    _, fold_metrics, _, _ = train_and_evaluate(
        df=df_fold,
        target=target_returns,
        known_reals=known_reals_r,
        unknown_reals=unknown_reals_r,
        train_cutoff=fold_cutoff,
        loss_fn=DirectionAwareQuantileLoss(quantile_weight=0.7, direction_weight=0.3),
        hidden_size=64,
        attention_head_size=4,
        hidden_continuous_size=32,
        max_epochs=15,  # fewer epochs per fold to save time
        patience=7,
        experiment_name=f"WF_fold{fold}",
    )
    fold_metrics['fold'] = fold
    wf_results.append(fold_metrics)

print("\n" + "=" * 60)
print("WALK-FORWARD SUMMARY")
print("=" * 60)
wf_df = pd.DataFrame(wf_results)
print(wf_df[['fold', 'dir_acc_15m', 'dir_acc_30m', 'dir_acc_45m', 'dir_acc_60m', 'mae', 'val_loss']].to_string(index=False))
print(f"\nMean direction accuracy:")
for h in [15, 30, 45, 60]:
    col = f'dir_acc_{h}m'
    if col in wf_df.columns:
        print(f"  {h}m: {wf_df[col].mean():.1%} (std: {wf_df[col].std():.1%})")

## 9. Results Comparison

In [ ]:
# Collect all experiment results
all_results = [metrics_a, metrics_b, metrics_c, metrics_d]
if metrics_e:
    all_results.append(metrics_e)

results_df = pd.DataFrame(all_results)
results_df = results_df.set_index('experiment')

# Display
display_cols = [c for c in results_df.columns if 'dir_acc' in c or c in ['mae', 'rmse', 'val_loss', 'epochs']]
print("\n" + "=" * 80)
print("EXPERIMENT COMPARISON")
print("=" * 80)
print(results_df[display_cols].to_string())

# Plot direction accuracy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Direction accuracy by horizon
dir_cols = [c for c in results_df.columns if 'dir_acc' in c]
if dir_cols:
    results_df[dir_cols].T.plot(kind='bar', ax=axes[0], rot=0)
    axes[0].set_title('Direction Accuracy by Horizon')
    axes[0].set_ylabel('Accuracy')
    axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random (50%)')
    axes[0].legend(loc='lower right', fontsize=8)
    axes[0].set_ylim(0.35, 0.75)

# MAE comparison
if 'mae' in results_df.columns:
    results_df['mae'].plot(kind='bar', ax=axes[1], color='steelblue')
    axes[1].set_title('Mean Absolute Error')
    axes[1].set_ylabel('MAE')
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('experiment_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Improvement over baseline
print("\n" + "=" * 80)
print("IMPROVEMENT OVER BASELINE")
print("=" * 80)
baseline_row = results_df.loc['A_baseline']
for exp_name in results_df.index:
    if exp_name == 'A_baseline':
        continue
    row = results_df.loc[exp_name]
    print(f"\n{exp_name}:")
    for col in dir_cols:
        horizon = col.replace('dir_acc_', '')
        delta = row[col] - baseline_row[col]
        print(f"  {horizon}: {delta:+.1%}")

## 10. Next Steps

Based on the experiment results above, choose the winning configuration and:

1. **If returns target wins**: Update `pipeline/train.py` to use `close_return` as target, and update `main.py` prediction logic to convert cumulative returns back to prices
2. **If direction loss wins**: Add `DirectionAwareQuantileLoss` to the training container
3. **If larger model wins**: Update `run_pipeline.py` defaults (`hidden_size=64`, `attention_head_size=4`)
4. **If ensemble wins**: Modify `main.py` to load and average top-3 checkpoints

### Tuning the direction loss weight
If direction-aware loss shows promise, try different weight ratios:

In [ ]:
# Optional: Sweep direction_weight to find optimal balance
# Uncomment and run if Experiment C/D show improvement

# sweep_results = []
# for dw in [0.1, 0.2, 0.3, 0.4, 0.5]:
#     print(f"\nDirection weight = {dw}")
#     _, m, _, _ = train_and_evaluate(
#         df=df_returns, target=target_returns,
#         known_reals=known_reals_r, unknown_reals=unknown_reals_r,
#         train_cutoff=train_cutoff_r,
#         loss_fn=DirectionAwareQuantileLoss(quantile_weight=1-dw, direction_weight=dw),
#         hidden_size=64, attention_head_size=4, hidden_continuous_size=32,
#         max_epochs=15, patience=7,
#         experiment_name=f"sweep_dw{dw}",
#     )
#     m['direction_weight'] = dw
#     sweep_results.append(m)

# sweep_df = pd.DataFrame(sweep_results)
# print(sweep_df[['direction_weight', 'dir_acc_15m', 'dir_acc_30m', 'dir_acc_60m', 'val_loss']].to_string(index=False))

print("Sweep code ready (uncomment to run).")